# 🏭 Phase 0: Business Understanding & Data Architecture
**Project:** Manufacturing 360 — Từ dữ liệu rời rạc đến quyết định điều hành (HPT Test)

---

## 1. Bối cảnh Kinh doanh (Business Context)

Doanh nghiệp sản xuất đang đối mặt với bài toán **"Data Silos" (Ốc đảo dữ liệu)**. Các phòng ban (Sales, Kho, Sản xuất, Nhân sự, QA) đang sử dụng các hệ thống độc lập. Hậu quả là Ban Lãnh đạo không thể trả lời nhanh những câu hỏi liên phòng ban, ví dụ: *"Đơn hàng trễ là do thiếu vật tư hay do công nhân xưởng A làm việc kém hiệu quả?"*

**Mục tiêu cốt lõi:**
Xây dựng một "Control Tower" (Tháp điều khiển) — dashboard tương tác giúp Ban điều hành nhìn thấu suốt toàn bộ chuỗi cung ứng:

```
Nhận đơn → Lấy vật tư → Đưa vào sản xuất → Kiểm định chất lượng → Giao hàng
```

---

## 2. Bản đồ Dữ liệu (The 6 Data Domains)

| # | Domain | Bảng | Mục tiêu |
|---|---|---|---|
| 1 | **CRM** (Khách hàng & Nhu cầu) | Customers, DemandForecast | Đánh giá tiềm năng khách hàng, dự báo nhu cầu thị trường |
| 2 | **Sales** (Đơn hàng & Sản phẩm) | SalesOrders, OrderLines, Products | Theo dõi trạng thái đơn hàng thực tế so với cam kết (`RequiredDate`) |
| 3 | **Production** (Điều hành Sản xuất) — *nút thắt trung tâm* | ProductionOrders, WorkCenters, BillOfMaterials | Lệnh sản xuất chạy ở xưởng nào; BOM quyết định việc kéo vật tư từ kho |
| 4 | **Materials** (Vật tư & Kho) | Materials, Suppliers, InventoryTransactions | Kiểm soát tồn kho, đánh giá rủi ro đứt gãy cung ứng theo `LeadTimeDays` |
| 5 | **Quality** (Kiểm định Chất lượng) | QualityInspections | Cảnh báo tỷ lệ phế phẩm/lỗi, nhu cầu rework, lãng phí năng lực |
| 6 | **HRM** (Nhân lực) | Employees, SkillsMatrix, WorkSchedules | Đánh giá năng lực, giờ làm thực tế tại các phân xưởng |

---

## 3. Bảng Tổng hợp Đặc trưng Từng Sheet (Data Profiling)

> Ghi chú kiểu dữ liệu dựa trên khảo sát thực tế file mẫu. Các cột ngày tháng hiện đang ở dạng **text**, cần convert sang `datetime` ở Phase 1.

### 3.1. CRM_Data.xlsx

**Sheet `Customers`** — 20 dòng, 9 cột — *Dimension khách hàng*

| Cột | Kiểu | Vai trò |
|---|---|---|
| CustomerID | text (PK) | Khóa nối SalesOrders, DemandForecast |
| CustomerName | text | Hiển thị báo cáo |
| Industry | text (19 giá trị) | Phân khúc theo ngành |
| Region | text (4 giá trị) | Lọc/phân tích theo khu vực |
| CustomerTier | text (Gold/Platinum/Silver...) | Phân hạng khách hàng |
| CreditLimit | int | Đánh giá rủi ro tài chính |
| AccountManagerID | text (FK) | Nối Employees — ai phụ trách khách hàng nào |
| ContractStartDate | text (ngày) | Vòng đời hợp đồng |
| ContractEndDate | text (ngày) | Cảnh báo hợp đồng sắp hết hạn |

**Sheet `DemandForecast`** — 330 dòng, 7 cột — *Dự báo nhu cầu*

| Cột | Kiểu | Vai trò |
|---|---|---|
| ForecastID | text (PK) | Định danh |
| CustomerID | text (FK) | Nối Customers |
| ProductID | text (FK) | Nối Products |
| ForecastMonth | text "yyyy-mm" | Trục thời gian dự báo |
| ForecastQty | int | Số lượng dự báo |
| ConfidenceLevel | text (Low/Medium/High) | Độ tin cậy — dùng để weight khi so với năng lực |
| ForecastType | text (Contractual/Estimated/Historical) | Nguồn gốc dự báo |

---

### 3.2. ERP_Sales_Data.xlsx

**Sheet `SalesOrders`** — 100 dòng, 7 cột

| Cột | Kiểu | Vai trò |
|---|---|---|
| OrderID | text (PK) | Khóa chính đơn hàng |
| CustomerID | text (FK) | Nối Customers |
| OrderDate | text (ngày) | Ngày đặt hàng |
| RequiredDate | text (ngày) | Cam kết giao hàng |
| OrderStatus | text (5 giá trị) | Confirmed/In Production/Shipped... |
| Priority | text (Low/Medium/High/Critical) | Mức ưu tiên — cốt lõi cho insight liên miền |
| SalesRepID | text (FK) | Nối Employees |

**Sheet `OrderLines`** — 306 dòng, 7 cột

| Cột | Kiểu | Vai trò |
|---|---|---|
| OrderLineID | text (PK) | Định danh dòng |
| OrderID | text (FK) | Nối SalesOrders (1–nhiều) |
| ProductID | text (FK) | Nối Products |
| Quantity | int | Số lượng đặt |
| UnitPrice | int | Đơn giá |
| LineTotal | int | Doanh thu dòng — kiểm tra khớp Quantity×UnitPrice |
| PromisedDate | text (ngày) | Cam kết giao theo dòng, đối chiếu ProductionOrder |

**Sheet `Products`** — 30 dòng, 8 cột

| Cột | Kiểu | Vai trò |
|---|---|---|
| ProductID | text (PK) | Danh mục sản phẩm |
| ProductName / ProductCategory | text | Phân nhóm (24 category) |
| UnitPrice / UnitCost | int | Tính biên lợi nhuận |
| LeadTimeDays | int | Thời gian sản xuất chuẩn |
| SafetyStock | int | Mức tồn kho an toàn |
| WorkCenterID | text (FK) | Nối WorkCenters |

⚠️ Doanh thu phải tính từ `OrderLines` (mức chi tiết nhất), không nhân theo số dòng ở `SalesOrders` để tránh nhân bản.

---

### 3.3. ERP_Production_Data.xlsx

**Sheet `ProductionOrders`** — 80 dòng, 12 cột — *bảng trung tâm*

| Cột | Kiểu | Vai trò |
|---|---|---|
| ProductionOrderID | text (PK) | Khóa chính |
| ProductID | text (FK) | Nối Products |
| SalesOrderID | text (FK, **22 null**) | Null = sản xuất tồn kho (make-to-stock) |
| PlannedQty / CompletedQty | int | Tính tỷ lệ hoàn thành |
| PlannedStartDate / PlannedEndDate | text (ngày) | Kế hoạch |
| ActualStartDate (**14 null**) / ActualEndDate (**28 null**) | text (ngày) | Null = chưa bắt đầu/chưa hoàn tất |
| Status | text (5 giá trị) | Planned/Released/Completed... |
| Priority | text (Low/Medium/Critical...) | Đối chiếu Priority của SalesOrders |
| WorkCenterID | text (FK) | Nối WorkCenters |

**Sheet `WorkCenters`** — 8 dòng, 7 cột

| Cột | Kiểu | Vai trò |
|---|---|---|
| WorkCenterID | text (PK) | Định danh trạm sản xuất |
| WorkCenterName / Department | text | Phân loại |
| CapacityPerDay | int | Năng lực tối đa/ngày |
| CostPerHour | int | Chi phí vận hành |
| EfficiencyRate | float (0–1) | Hiệu suất thực tế |
| SupervisorID | text (FK) | Nối Employees |

**Sheet `BillOfMaterials`** — 176 dòng, 5 cột

| Cột | Kiểu | Vai trò |
|---|---|---|
| BOMID | text (PK) | Định danh |
| ProductID / MaterialID | text (FK, **quan hệ nhiều-nhiều**) | Cẩn thận khi join, tránh nhân bản |
| QuantityRequired | float | Định mức vật tư/1 sản phẩm |
| ScrapFactor | float (0–1) | Hao hụt — nhu cầu thực = QuantityRequired×(1+ScrapFactor) |

→ BOM là cầu nối then chốt để trả lời Q2 (Production–Materials).

---

### 3.4. ERP_Materials_Data.xlsx

**Sheet `Materials`** — 50 dòng, 10 cột

| Cột | Kiểu | Vai trò |
|---|---|---|
| MaterialID | text (PK) | Định danh vật tư |
| MaterialName / MaterialCategory / UnitOfMeasure | text | Mô tả |
| UnitCost | int | Tính giá trị tồn kho |
| SupplierID | text (FK) | Nối Suppliers |
| LeadTimeDays | int | Thời gian chờ hàng |
| MinOrderQty | int | Ràng buộc đặt hàng |
| CurrentStock | int | Tồn kho hiện tại |
| ReorderPoint | int | Điểm đặt hàng lại — cốt lõi cho cảnh báo rủi ro |

**Sheet `Suppliers`** — 15 dòng, 7 cột

| Cột | Kiểu | Vai trò |
|---|---|---|
| SupplierID | text (PK) | Định danh |
| SupplierName / ContactPerson / Country | text | Thông tin liên hệ |
| Rating | text (A/B/C) | Đánh giá chất lượng nhà cung cấp |
| PaymentTerms | text (Net 30/45/60) | Điều khoản thanh toán |
| LeadTimeAvg | int | Thời gian giao hàng trung bình |

**Sheet `InventoryTransactions`** — 742 dòng, 7 cột

| Cột | Kiểu | Vai trò |
|---|---|---|
| TransactionID | text (PK) | Định danh giao dịch |
| MaterialID | text (FK) | Nối Materials |
| TransactionDate | text (ngày) | Trục thời gian |
| TransactionType | text (Receipt/Issue/Adjustment) | Loại giao dịch |
| Quantity | int (**có giá trị âm**) | Âm=xuất/điều chỉnh giảm, dương=nhập |
| ReferenceDoc | text | Chứng từ tham chiếu |
| WarehouseLocation | text (5 kho) | Phân tích theo kho |

---

### 3.5. Quality_Data.xlsx

**Sheet `QualityInspections`** — 118 dòng, 10 cột

| Cột | Kiểu | Vai trò |
|---|---|---|
| InspectionID | text (PK) | Định danh |
| ProductionOrderID | text (FK) | Nối ProductionOrders |
| ProductID | text (FK) | Nối Products |
| InspectionDate | text (ngày) | Trục thời gian |
| InspectorID | text (FK) | Nối Employees |
| SampleSize | int | Cỡ mẫu kiểm tra |
| DefectsFound | int | Số lỗi — tính tỷ lệ lỗi = DefectsFound/SampleSize |
| DefectType | text (**26 null** khi Pass) | Functional/Dimensional/Surface |
| InspectionResult | text (Pass/Fail/Conditional) | Kết quả kiểm tra |
| ReworkRequired | bool | Có cần làm lại không |

---

### 3.6. HRM_Data.xlsx

**Sheet `Employees`** — 50 dòng, 10 cột

| Cột | Kiểu | Vai trò |
|---|---|---|
| EmployeeID | text (PK) | Định danh |
| FirstName / LastName | text | Hiển thị |
| Department / Position | text | 9 phòng ban, 19 vị trí |
| HireDate | text (ngày) | Thâm niên |
| Salary | int | Chi phí nhân sự |
| SkillLevel | text (Junior→Expert) | Đánh giá năng lực |
| WorkCenterID | text (FK, **26 null**) | Null = nhân viên ngoài khối sản xuất |
| ShiftPattern | text (Day/Night) | Ca làm việc |

**Sheet `SkillsMatrix`** — 174 dòng, 6 cột

| Cột | Kiểu | Vai trò |
|---|---|---|
| SkillRecordID | text (PK) | Định danh |
| EmployeeID | text (FK) | Nối Employees (1–nhiều) |
| SkillName | text (12 loại) | Tên kỹ năng |
| ProficiencyLevel | text (Beginner→Advanced) | Mức độ thành thạo |
| CertificationDate | text (ngày) | Ngày chứng nhận |
| ExpiryDate | text (**87 null**) | Null = chứng chỉ không hết hạn |

**Sheet `WorkSchedules`** — 1300 dòng, 10 cột — *mức chi tiết theo ngày*

| Cột | Kiểu | Vai trò |
|---|---|---|
| ScheduleID | text (PK) | Định danh |
| EmployeeID | text (FK, chỉ 20/50 nhân viên có lịch) | Chỉ áp dụng nhân sự theo ca |
| ScheduleDate | text (ngày) | Trục thời gian |
| ShiftType / StartTime / EndTime | text | Thông tin ca làm |
| PlannedHours | int (luôn = 8) | Giờ kế hoạch cố định |
| ActualHours | float (**119 null**) | Null có thể do vắng/nghỉ |
| OvertimeHours | float | Giờ tăng ca |
| Status | text (Scheduled/Absent/Leave...) | Trạng thái chấm công |

---

## 4. Các Câu hỏi Kinh doanh Liên miền (Cross-Domain Business Questions)

### Q1. Sales vs. Production — Đơn hàng ưu tiên có nguy cơ trễ hẹn
**Câu hỏi:** Những đơn hàng có độ ưu tiên cao nào đang đứng trước nguy cơ trễ hẹn do lệnh sản xuất tương ứng bị chậm?

**Công thức đo lường:**
- Join `SalesOrders` ↔ `ProductionOrders` qua `SalesOrderID`.
- Độ trễ = `ActualEndDate − PlannedEndDate` (nếu `ActualEndDate` null → dùng ngày hiện tại nếu `Status ≠ Completed` và đã quá `PlannedEndDate`, gắn cờ "At Risk").
- Lọc `SalesOrders.Priority` ∈ {High, Critical}.

### Q2. Production vs. Materials — Vật tư chặn tiến độ sản xuất
**Câu hỏi:** Lệnh sản xuất nào không thể khởi chạy đúng hạn do vật tư dưới mức an toàn và nhà cung cấp giao chậm?

**Công thức đo lường:**
- Với mỗi `ProductionOrder` có `Status ∈ {Planned, Released}`, join `BillOfMaterials` theo `ProductID` → lấy danh sách `MaterialID` cần dùng.
- So `Materials.CurrentStock` với `ReorderPoint`.
- Nếu thiếu: ngày có hàng dự kiến = hôm nay + `Suppliers.LeadTimeAvg` (qua `Materials.SupplierID`) → so với `PlannedStartDate`.

### Q3. Production vs. Quality — Rework tập trung ở đâu
**Câu hỏi:** Tỷ lệ hàng lỗi đang tập trung cao bất thường ở work center nào, lãng phí bao nhiêu vật tư?

**Công thức đo lường:**
- Join `QualityInspections` ↔ `ProductionOrders` qua `ProductionOrderID` để lấy `WorkCenterID`.
- Tỷ lệ rework = COUNT(`ReworkRequired=True`) / COUNT(inspection) theo từng `WorkCenterID`.
- Lãng phí vật tư ước tính = `BillOfMaterials.QuantityRequired × Materials.UnitCost` của các lệnh bị rework.

### Q4. Production vs. HRM — Con người đứng sau vấn đề hiệu suất
**Câu hỏi:** Work center có tỷ lệ trễ/rework cao (từ Q1, Q3) có tương quan với tỷ lệ vắng mặt cao hoặc thiếu nhân sự bậc cao không?

**Công thức đo lường:**
- Join `Employees.WorkCenterID` để đếm % `SkillLevel = Expert/Senior` theo work center.
- Join `WorkSchedules` (qua `EmployeeID` → `Employees.WorkCenterID`) để tính tỷ lệ `Status = Absent` và tổng `OvertimeHours` theo work center.
- Đối chiếu 2 chỉ số này với danh sách work center trễ nhiều nhất (từ Q1) và rework cao nhất (từ Q3).

---

## 5. Chiến lược Tích hợp AI (AI Integration Strategy)

Sử dụng **OpenRouter API (Qwen Model)** để nâng cấp Dashboard từ "Phân tích tĩnh" sang "Trợ lý ảo điều hành":

- **Automated Root-Cause Analysis:** Tự động tìm nguyên nhân gốc (VD: AI phát hiện Đơn hàng #101 trễ là do Xưởng B thiếu công nhân bậc cao và Vật tư Z bị chậm).
- **Executive Summary:** AI tự động sinh ra 3 gạch đầu dòng báo cáo tình hình mỗi ngày cho Giám đốc dựa trên bảng dữ liệu aggregated.

**Cơ chế kỹ thuật:**
Sau khi tầng BI tính xong các bảng tổng hợp (VD: `late_orders_summary`, `rework_by_workcenter`, `material_risk`), hệ thống gửi các bảng này dưới dạng JSON vào prompt gọi Qwen (qua OpenRouter), yêu cầu trả về JSON có cấu trúc:

```json
{
  "finding": "...",
  "root_cause": "...",
  "priority": "High | Medium | Low",
  "recommended_action": "..."
}
```

**Nguyên tắc quan trọng:** AI **không tự tính toán số liệu** — chỉ diễn giải các số liệu đã được tính sẵn ở tầng dữ liệu (SQL/DAX/pandas). Điều này đảm bảo tính đúng đắn của kết quả nằm ở tầng dữ liệu có thể kiểm chứng, còn AI chỉ đóng vai trò chuyển hóa số liệu thành ngôn ngữ tự nhiên dễ hiểu cho Ban điều hành.

---

## 6. Giả định & Hạn chế Dữ liệu (Assumptions & Limitations)

| # | Vấn đề | Vị trí | Giả định/Xử lý |
|---|---|---|---|
| 1 | Ngày tháng đang là kiểu `text` | Hầu hết mọi sheet | Convert sang `datetime` ở Phase 1 trước khi tính kỳ hạn/độ trễ |
| 2 | `SalesOrderID` null (22/80 dòng) | ProductionOrders | Giả định: sản xuất tồn kho (make-to-stock), không phải lỗi dữ liệu |
| 3 | `ActualStartDate`/`ActualEndDate` null | ProductionOrders | Coi là "chưa bắt đầu"/"chưa hoàn thành", không tính vào độ trễ đã xảy ra |
| 4 | `DefectType` null khi Pass | QualityInspections | Hợp lệ theo logic nghiệp vụ (Pass không có loại lỗi) |
| 5 | `WorkCenterID` null ở Employees (26/50) | HRM | Nhân sự không thuộc khối sản xuất (Sales, HR, Admin...) |
| 6 | `Quantity` âm/dương | InventoryTransactions | Cần xác nhận convention (âm=xuất, dương=nhập) trước khi cộng dồn tồn kho |
| 7 | `ActualHours` null (119/1300) | WorkSchedules | Đối chiếu cột `Status` (Absent/Leave) để xác nhận không phải lỗi dữ liệu |
| 8 | Quan hệ nhiều-nhiều Product↔Material | BillOfMaterials | Cẩn thận khi join, tránh nhân bản số lượng ở tầng Product hoặc Material |
| 9 | `EmployeeID` trong WorkSchedules chỉ phủ 20/50 nhân viên | HRM | Giả định: chỉ nhân sự làm việc theo ca (production/quality) mới có lịch |

---

## 7. Success Criteria (Tiêu chí thành công của dự án)

- [ ] Data model tích hợp đủ 6 domain, quan hệ khóa chính xác, không nhân bản số liệu khi tổng hợp.
- [ ] Dashboard trả lời được đầy đủ Q1–Q4 trong tối đa 2–3 lần click filter.
- [ ] Có tối thiểu 3 insight định lượng, mỗi insight nêu rõ: hiện tượng — nguyên nhân — mức ưu tiên — hành động đề xuất.
- [ ] Có ít nhất 1 phát hiện liên miền rõ ràng (không chỉ mô tả từng hệ thống riêng lẻ).
- [ ] AI Root-cause/Executive Summary hoạt động dựa trên dữ liệu đã tổng hợp sẵn (không "ảo giác" số liệu).
- [ ] Toàn bộ giả định/hạn chế dữ liệu được công bố minh bạch trong báo cáo.